In [1]:
import sys
import subprocess

# List of required packages
packages = [
    "pandas",
    "numpy",
    "pyarrow",
    "tqdm",
    "scikit-learn",
    "torch"
]

# Install all packages
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

0

Imports + Seed (Reproducibility)

In [2]:
import pandas as pd
import numpy as np
import re
import pyarrow.parquet as pq
from tqdm import tqdm
import random
import torch

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

# Torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Fix seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [3]:
file_path = "dataset_10M.parquet" 
parquet_file = pq.ParquetFile(file_path)

In [4]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

In [5]:
def load_subset(limit=300000):
    texts, labels = [], []
    count = 0

    for batch in parquet_file.iter_batches(batch_size=50000):
        df = batch.to_pandas()

        df["DATA"] = df["DATA"].apply(preprocess)

        texts.extend(df["DATA"].tolist())
        labels.extend(df["TOPIC"].tolist())

        count += len(df)
        if count >= limit:
            break

    return texts, labels

texts, labels = load_subset()
print("Loaded:", len(texts))

Loaded: 300000


In [6]:
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

In [7]:
# CELL 6: Feature Extraction (FIXED for Naive Bayes)

vectorizer = HashingVectorizer(
    n_features=2**18,
    alternate_sign=False   # 🔥 IMPORTANT FIX
)

X = vectorizer.transform(texts)
y = labels_encoded

In [8]:
# Models
nb_model = MultinomialNB()
lr_model = SGDClassifier(loss="log_loss")
svm_model = SGDClassifier(loss="hinge")

# Train
nb_model.fit(X, y)
lr_model.fit(X, y)
svm_model.fit(X, y)

SGDClassifier()

In [9]:
# CELL 8.5: TF-IDF + Logistic Regression (OPTIMIZED VERSION)

from sklearn.feature_extraction.text import TfidfVectorizer

print("\nTraining TF-IDF (Optimized for speed and memory)...\n")

# 🔥 Optimized TF-IDF (safe for i3 laptop)
tfidf = TfidfVectorizer(
    max_features=30000,     # reduced from 100000
    ngram_range=(1,1),      # remove bigrams (huge speed boost)
    min_df=5,               # ignore rare words
    max_df=0.9              # ignore very common words
)

# 🔥 Use smaller subset to avoid memory issues
texts_small = texts[:100000]
y_small = y[:100000]

# Fit and transform
X_tfidf = tfidf.fit_transform(texts_small)

# Train Logistic Regression (SGD)
lr_tfidf = SGDClassifier(loss="log_loss", max_iter=1000)

lr_tfidf.fit(X_tfidf, y_small)

# Evaluate
preds = lr_tfidf.predict(X_tfidf)

tfidf_acc = accuracy_score(y_small, preds)
tfidf_f1 = f1_score(y_small, preds, average="weighted")

print("TF-IDF + Logistic Regression (Optimized):")
print(f"Accuracy: {tfidf_acc:.4f}")
print(f"F1 Score: {tfidf_f1:.4f}")


Training TF-IDF (Optimized for speed and memory)...

TF-IDF + Logistic Regression (Optimized):
Accuracy: 0.8634
F1 Score: 0.8556


In [10]:
from collections import Counter

def build_vocab(texts, max_vocab=50000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {word: i+1 for i, (word, _) in enumerate(counter.most_common(max_vocab))}
    return vocab

vocab = build_vocab(texts)

In [11]:
def text_to_sequence(text, vocab, max_len=100):
    seq = [vocab.get(word, 0) for word in text.split()]
    
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    
    return seq

In [12]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        seq = text_to_sequence(self.texts[idx], self.vocab)
        return torch.tensor(seq), torch.tensor(self.labels[idx])

In [13]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 128, 3)
        self.conv2 = nn.Conv1d(embed_dim, 128, 4)
        self.conv3 = nn.Conv1d(embed_dim, 128, 5)

        self.fc = nn.Linear(128*3, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x1 = torch.relu(self.conv1(x)).max(dim=2)[0]
        x2 = torch.relu(self.conv2(x)).max(dim=2)[0]
        x3 = torch.relu(self.conv3(x)).max(dim=2)[0]

        x = torch.cat([x1, x2, x3], dim=1)
        x = self.dropout(x)

        return self.fc(x)

In [14]:
dataset = TextDataset(texts, y, vocab)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_cnn = TextCNN(len(vocab)+1, 128, len(le.classes_)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

In [16]:
epochs = 3

for epoch in range(epochs):
    model_cnn.train()
    total_loss = 0

    for X_batch, y_batch in tqdm(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model_cnn(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")

100%|████████████████████████████████████████████████████████████████████████████████| 938/938 [17:25<00:00,  1.11s/it]


Epoch 1, Loss: 923.6250666379929


100%|████████████████████████████████████████████████████████████████████████████████| 938/938 [09:59<00:00,  1.56it/s]


Epoch 2, Loss: 591.0573028326035


100%|████████████████████████████████████████████████████████████████████████████████| 938/938 [10:10<00:00,  1.54it/s]

Epoch 3, Loss: 477.89801639318466


In [17]:
model_cnn.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = model_cnn(X_batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

cnn_acc = accuracy_score(all_labels, all_preds)
cnn_f1 = f1_score(all_labels, all_preds, average="weighted")

print("CNN:", cnn_acc, cnn_f1)

CNN: 0.8770166666666667 0.8729176835580171


In [18]:
# Classical Models using Hashing Features

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

print("\nTraining Classical Models (Hashing)...\n")

# Models
nb_model = MultinomialNB()
lr_model = SGDClassifier(loss="log_loss", max_iter=1000)
svm_model = SGDClassifier(loss="hinge", max_iter=1000)

# Train
nb_model.fit(X, y)
lr_model.fit(X, y)
svm_model.fit(X, y)

# Evaluate
def evaluate(model, X, y):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, average="weighted")
    return acc, f1

nb_acc, nb_f1 = evaluate(nb_model, X, y)
lr_acc, lr_f1 = evaluate(lr_model, X, y)
svm_acc, svm_f1 = evaluate(svm_model, X, y)

print(f"Naive Bayes: {nb_acc:.4f}, F1: {nb_f1:.4f}")
print(f"Logistic Regression (Hashing): {lr_acc:.4f}, F1: {lr_f1:.4f}")
print(f"SVM: {svm_acc:.4f}, F1: {svm_f1:.4f}")


Training Classical Models (Hashing)...

Naive Bayes: 0.7999, F1: 0.7902
Logistic Regression (Hashing): 0.8020, F1: 0.7923
SVM: 0.8754, F1: 0.8707


In [19]:
print("\nFinal Comparison:\n")

print(f"Naive Bayes: {nb_acc:.4f}, F1: {nb_f1:.4f}")
print(f"Logistic Regression (Hashing): {lr_acc:.4f}, F1: {lr_f1:.4f}")
print(f"SVM: {svm_acc:.4f}, F1: {svm_f1:.4f}")

# Only print if exists
try:
    print(f"TF-IDF + LR: {tfidf_acc:.4f}, F1: {tfidf_f1:.4f}")
except:
    pass

print(f"CNN: {cnn_acc:.4f}, F1: {cnn_f1:.4f}")


Final Comparison:

Naive Bayes: 0.7999, F1: 0.7902
Logistic Regression (Hashing): 0.8020, F1: 0.7923
SVM: 0.8754, F1: 0.8707
TF-IDF + LR: 0.8634, F1: 0.8556
CNN: 0.8770, F1: 0.8729


In [21]:
# FINAL CELL: Save all models and metadata

import torch
import json
import os
import joblib

# Create directory
os.makedirs("final_models", exist_ok=True)

# =========================
# Save CNN model
# =========================
torch.save(model_cnn.state_dict(), "final_models/cnn_model.pth")

# =========================
# Save label encoder
# =========================
try:
    with open("final_models/label_encoder.json", "w") as f:
        json.dump(label_encoder.classes_.tolist(), f)
except:
    print("Label encoder not found")

# =========================
# Save vocabulary (if used)
# =========================
try:
    with open("final_models/vocab.json", "w") as f:
        json.dump(vocab, f)
except:
    print("Vocab not found or not used")

# =========================
# Save classical models
# =========================
try:
    joblib.dump(nb_model, "final_models/nb_model.pkl")
    joblib.dump(lr_model, "final_models/lr_model.pkl")
    joblib.dump(svm_model, "final_models/svm_model.pkl")
except:
    print("Some classical models not found")

# =========================
# Save TF-IDF model (optional)
# =========================
try:
    joblib.dump(tfidf, "final_models/tfidf_vectorizer.pkl")
    joblib.dump(lr_tfidf, "final_models/lr_tfidf_model.pkl")
except:
    print("TF-IDF model not found")

print("\n All models and metadata saved successfully in 'final_models/' folder!")

Label encoder not found

 All models and metadata saved successfully in 'final_models/' folder!
